# Apache Polaris — RBAC Entity Graph Visualization

**Date:** 2026-05-15  
**Reference:** https://polaris.apache.org/releases/1.3.0/entities/

## 학습 목표 (Learning Goals)
- Polaris 엔티티 간의 관계를 그래프로 시각화
- Principal → Principal Role → Catalog Role → Privilege 체인 확인
- 실제 설정된 RBAC 구조를 한눈에 파악

In [31]:
# ============================================================
# Prerequisites
# ============================================================
# Run this cell first before any other cells.
#
# Recommended (uv):
#   uv add requests pandas pyvis
#
# Alternative (pip):
#   pip install requests pandas pyvis
# ============================================================

import subprocess
import sys
import os

REQUIRED = ['requests', 'pandas', 'pyvis']

def install(pkg):
    # Try uv first
    result = subprocess.run(
        ['uv', 'add', pkg],
        capture_output=True, text=True,
        cwd=os.getcwd()
    )
    if result.returncode == 0:
        return '✅', 'uv'
    
    # Fall back to pip
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        return '✅', 'pip'
    
    return '❌', 'failed'

print('Installing dependencies...')
for pkg in REQUIRED:
    status, method = install(pkg)
    print(f'  {status} {pkg} ({method})')

print('\n⚠️  Restart kernel after first install.')
print('   Kernel menu → Restart Kernel → Run All Cells Below')

Installing dependencies...
  ✅ requests (uv)
  ✅ pandas (uv)
  ✅ pyvis (uv)

⚠️  Restart kernel after first install.
   Kernel menu → Restart Kernel → Run All Cells Below


In [37]:
# Cell 1 — Setup: Polaris connection
import requests
import json
import pandas as pd
from IPython.display import display, HTML
from pyvis.network import Network

# ========================================
# Configuration — update if needed
# ========================================
POLARIS_URL = 'http://192.168.139.2:8181'
REALM = 'DATACORP-PROD'
ROOT_CLIENT_ID = 'datacorp-root'
ROOT_CLIENT_SECRET = 'datacorp-secret'

def get_token(client_id=ROOT_CLIENT_ID, client_secret=ROOT_CLIENT_SECRET):
    r = requests.post(
        f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
        headers={'Polaris-Realm': REALM},
        data={
            'grant_type': 'client_credentials',
            'client_id': client_id,
            'client_secret': client_secret,
            'scope': 'PRINCIPAL_ROLE:ALL'
        }
    )
    return r.json()['access_token']

def headers(token):
    return {
        'Authorization': f'Bearer {token}',
        'Polaris-Realm': REALM,
        'Content-Type': 'application/json'
    }

TOKEN = get_token()
print(f'✅ Connected to Polaris at {POLARIS_URL}')

✅ Connected to Polaris at http://192.168.139.2:8181


In [38]:
# Cell 2 — Fetch all entities from Polaris API

def safe_get(url):
    try:
        r = requests.get(url, headers=headers(TOKEN))
        return r.json() if r.status_code == 200 else {}
    except:
        return {}

# Fetch principals
principals_data = safe_get(f'{POLARIS_URL}/api/management/v1/principals')
principals = principals_data.get('principals', [])

# Fetch principal roles
principal_roles_data = safe_get(f'{POLARIS_URL}/api/management/v1/principal-roles')
principal_roles = principal_roles_data.get('roles', [])

# Fetch catalogs
catalogs_data = safe_get(f'{POLARIS_URL}/api/management/v1/catalogs')
catalogs = catalogs_data.get('catalogs', [])

print(f'📦 Principals: {len(principals)}')
print(f'👥 Principal Roles: {len(principal_roles)}')
print(f'📚 Catalogs: {len(catalogs)}')

# Fetch principal role assignments per principal
principal_role_assignments = {}  # principal -> [roles]
for p in principals:
    pname = p['name']
    r = safe_get(f'{POLARIS_URL}/api/management/v1/principals/{pname}/principal-roles')
    principal_role_assignments[pname] = [role['name'] for role in r.get('roles', [])]

# Fetch catalog roles per catalog
catalog_roles = {}  # catalog -> [catalog_roles]
for c in catalogs:
    cname = c['name']
    r = safe_get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles')
    catalog_roles[cname] = r.get('roles', [])

# Fetch principal role -> catalog role assignments
pr_catalog_role_assignments = {}  # principal_role -> {catalog -> [catalog_roles]}
for pr in principal_roles:
    prname = pr['name']
    pr_catalog_role_assignments[prname] = {}
    for c in catalogs:
        cname = c['name']
        r = safe_get(f'{POLARIS_URL}/api/management/v1/principal-roles/{prname}/catalog-roles/{cname}')
        roles = r.get('roles', [])
        if roles:
            pr_catalog_role_assignments[prname][cname] = [role['name'] for role in roles]

# Fetch grants per catalog role
catalog_role_grants = {}  # catalog/role -> [grants]
for cname, roles in catalog_roles.items():
    for role in roles:
        rname = role['name']
        key = f'{cname}/{rname}'
        r = safe_get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles/{rname}/grants')
        catalog_role_grants[key] = r.get('grants', [])

# Fetch namespaces and tables per catalog
namespaces = {}  # catalog -> [namespaces]
tables = {}      # catalog/namespace -> [tables]
for c in catalogs:
    cname = c['name']
    r = safe_get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces')
    ns_list = r.get('namespaces', [])
    namespaces[cname] = ns_list
    for ns in ns_list:
        ns_name = ns[0] if isinstance(ns, list) else ns
        r2 = safe_get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/tables')
        tables[f'{cname}/{ns_name}'] = r2.get('identifiers', [])

# Fetch views per namespace
views = {}  # catalog/namespace -> [views]
for c in catalogs:
    cname = c['name']
    for ns in namespaces.get(cname, []):
        ns_name = ns[0] if isinstance(ns, list) else ns
        r3 = safe_get(f'{POLARIS_URL}/api/catalog/v1/{cname}/namespaces/{ns_name}/views')
        views[f'{cname}/{ns_name}'] = r3.get('identifiers', [])

print('✅ All entity data fetched')

📦 Principals: 13
👥 Principal Roles: 11
📚 Catalogs: 5
✅ All entity data fetched


In [39]:
# Cell 3 — Build and render RBAC graph

net = Network(
    height='750px',
    width='100%',
    directed=True,
    notebook=True,
    bgcolor='#1a1a2e',
    font_color='white'
)

net.set_options("""
{
  "nodes": {
    "font": {"size": 14, "color": "white"},
    "borderWidth": 2,
    "shadow": true
  },
  "edges": {
    "arrows": {"to": {"enabled": true, "scaleFactor": 0.8}},
    "color": {"color": "#888888", "highlight": "#ffffff"},
    "font": {"size": 10, "color": "#aaaaaa", "align": "middle"},
    "smooth": {"type": "continuous"}
  },
  "physics": {
    "enabled": false
  },
  "interaction": {
    "hover": true,
    "tooltipDelay": 100,
    "dragNodes": true,
    "dragView": true,
    "zoomView": false
  }
}
""")

added_nodes = set()

def add_node(node_id, label, color, shape='dot', size=20, title=None):
    if node_id not in added_nodes:
        net.add_node(
            node_id,
            label=label,
            color=color,
            shape=shape,
            size=size,
            title=title or label
        )
        added_nodes.add(node_id)

def add_edge(src, dst, label='', color='#666666'):
    net.add_edge(src, dst, label=label, color=color)

# === Add Principals ===
for p in principals:
    pid = f"principal:{p['name']}"
    add_node(pid, f"👤 {p['name']}", '#4a90e2', shape='ellipse', size=25,
             title=f"Principal\nType: {p.get('type', 'SERVICE')}")

# === Add Principal Roles ===
for pr in principal_roles:
    prid = f"principal_role:{pr['name']}"
    add_node(prid, f"👥 {pr['name']}", '#27ae60', shape='box', size=20,
             title=f"Principal Role\n{pr['name']}")

# === Add Principal → Principal Role edges ===
for pname, role_names in principal_role_assignments.items():
    pid = f"principal:{pname}"
    for rname in role_names:
        prid = f"principal_role:{rname}"
        add_edge(pid, prid, 'assigned to', '#4a90e2')

# === Add Catalogs ===
for c in catalogs:
    cid = f"catalog:{c['name']}"
    storage = c.get('storageConfigInfo', {}).get('storageType', 'S3')
    add_node(cid, f"📚 {c['name']}", '#e67e22', shape='database', size=30,
             title=f"Catalog\nType: {c.get('type', 'INTERNAL')}\nStorage: {storage}")

# === Add Catalog Roles ===
for cname, roles in catalog_roles.items():
    cid = f"catalog:{cname}"
    for role in roles:
        rname = role['name']
        crid = f"catalog_role:{cname}/{rname}"
        add_node(crid, f"🔑 {rname}", '#e74c3c', shape='box', size=18,
                 title=f"Catalog Role\n{rname}\nin {cname}")
        add_edge(cid, crid, 'has role', '#e67e22')

# === Add Principal Role → Catalog Role edges ===
for prname, catalog_assignments in pr_catalog_role_assignments.items():
    prid = f"principal_role:{prname}"
    for cname, role_names in catalog_assignments.items():
        for rname in role_names:
            crid = f"catalog_role:{cname}/{rname}"
            add_edge(prid, crid, 'granted', '#27ae60')

# === Add Privileges ===
for key, grants in catalog_role_grants.items():
    crid = f"catalog_role:{key}"
    for grant in grants:
        priv = grant.get('privilege', 'UNKNOWN')
        scope = grant.get('type', '')
        pvid = f"privilege:{key}/{priv}"
        add_node(pvid, f"⚡ {priv}", '#9b59b6', shape='triangle', size=15,
                 title=f"Privilege\n{priv}\nScope: {scope}")
        add_edge(crid, pvid, 'grants', '#9b59b6')

# === Add Namespaces ===
for cname, ns_list in namespaces.items():
    cid = f"catalog:{cname}"
    for ns in ns_list:
        ns_name = ns[0] if isinstance(ns, list) else ns
        nsid = f"namespace:{cname}/{ns_name}"
        add_node(nsid, f"📁 {ns_name}", '#f1c40f', shape='dot', size=18,
                 title=f"Namespace\n{ns_name}\nin {cname}")
        add_edge(cid, nsid, 'contains', '#f1c40f')

# === Add Tables ===
for key, table_list in tables.items():
    cname, ns_name = key.split('/', 1)
    nsid = f"namespace:{key}"
    for t in table_list:
        tname = t.get('name', str(t)) if isinstance(t, dict) else str(t)
        tid = f"table:{key}/{tname}"
        add_node(tid, f"📊 {tname}", '#95a5a6', shape='dot', size=14,
                 title=f"Table\n{tname}\nin {ns_name}")
        add_edge(nsid, tid, 'contains', '#95a5a6')

# === Add Views ===
for key, view_list in views.items():
    cname, ns_name = key.split('/', 1)
    nsid = f'namespace:{key}'
    for v in view_list:
        vname = v.get('name', str(v)) if isinstance(v, dict) else str(v)
        vid = f'view:{key}/{vname}'
        add_node(vid, f'👁️ {vname}', '#e056f5', shape='diamond', size=14,
                 title=f'View\n{vname}\nin {ns_name}')
        add_edge(nsid, vid, 'contains', '#e056f5')

print(f'✅ Graph built: {len(added_nodes)} nodes, {len(net.edges)} edges')

# Save and add zoom controls
html_file = 'polaris_rbac_graph.html'
net.save_graph(html_file)

# Inject zoom controls into the HTML
with open(html_file, 'r') as f:
    html = f.read()

zoom_controls = """
<div style="position:fixed; bottom:20px; right:20px; z-index:9999; display:flex; flex-direction:column; gap:8px;">
  <button onclick="zoomIn()" style="width:40px;height:40px;font-size:20px;cursor:pointer;border-radius:8px;border:none;background:#4a90e2;color:white;">+</button>
  <button onclick="zoomOut()" style="width:40px;height:40px;font-size:20px;cursor:pointer;border-radius:8px;border:none;background:#4a90e2;color:white;">−</button>
  <button onclick="fitGraph()" style="width:40px;height:40px;font-size:14px;cursor:pointer;border-radius:8px;border:none;background:#27ae60;color:white;">⊡</button>
</div>
<script>
function zoomIn() {
  var network = window.network || document.getElementById('mynetwork')._network;
  var scale = network.getScale();
  network.moveTo({scale: scale * 1.2});
}
function zoomOut() {
  var network = window.network || document.getElementById('mynetwork')._network;
  var scale = network.getScale();
  network.moveTo({scale: scale * 0.8});
}
function fitGraph() {
  var network = window.network || document.getElementById('mynetwork')._network;
  network.fit({animation: {duration: 500, easingFunction: 'easeInOutQuad'}});
}
</script>
"""

# Inject before closing body tag
html = html.replace('</body>', zoom_controls + '</body>')

with open(html_file, 'w') as f:
    f.write(html)

from IPython.display import IFrame
IFrame(html_file, width='100%', height='750px')

✅ Graph built: 97 nodes, 110 edges


## 그래프 조작 방법 (How to interact)

- **드래그**: 노드를 클릭하고 드래그하여 위치 변경
- **줌**: 마우스 휠로 확대/축소
- **클릭**: 노드 클릭 시 연결된 엣지 하이라이트
- **호버**: 노드에 마우스를 올리면 상세 정보 표시

## 그래프 범례 (Legend)
| 색상 | 엔티티 타입 |
|------|------------|
| 🔵 파란색 | Principal (사용자/서비스) |
| 🟢 초록색 | Principal Role (그룹) |
| 🟠 주황색 | Catalog (데이터 저장소) |
| 🔴 빨간색 | Catalog Role (카탈로그 권한 그룹) |
| 🟣 보라색 | Privilege (실제 권한) |
| 🟡 노란색 | Namespace |
| ⚫ 회색 | Table |

In [40]:
# Cell 4 — Summary table: Entity counts
summary = {
    'Entity Type': ['Principal', 'Principal Role', 'Catalog', 'Catalog Role', 'Namespace', 'Table'],
    'Count': [
        len(principals),
        len(principal_roles),
        len(catalogs),
        sum(len(roles) for roles in catalog_roles.values()),
        sum(len(ns) for ns in namespaces.values()),
        sum(len(t) for t in tables.values())
    ],
    'Color': ['🔵 Blue', '🟢 Green', '🟠 Orange', '🔴 Red', '🟡 Yellow', '⚫ Gray']
}
df = pd.DataFrame(summary)
print('## Polaris Entity Summary')
display(df)

## Polaris Entity Summary


,Entity Type,Count,Color
0,Principal,13,🔵 Blue
1,Principal Role,11,🟢 Green
2,Catalog,5,🟠 Orange
3,Catalog Role,19,🔴 Red
4,Namespace,14,🟡 Yellow
5,Table,0,⚫ Gray


In [41]:
# Cell 5 — Show principal role chain as text
print('## RBAC Chain Summary\n')
for pname, role_names in principal_role_assignments.items():
    print(f'Principal: {pname}')
    for rname in role_names:
        print(f'  └── Principal Role: {rname}')
        catalog_assignments = pr_catalog_role_assignments.get(rname, {})
        for cname, croles in catalog_assignments.items():
            for crname in croles:
                print(f'      └── Catalog Role: {crname} (in {cname})')
                key = f'{cname}/{crname}'
                grants = catalog_role_grants.get(key, [])
                for g in grants:
                    print(f'          └── Privilege: {g.get("privilege", "?")} [{g.get("type", "?")}]')
    print()

## RBAC Chain Summary

Principal: alice
  └── Principal Role: data-engineer
      └── Catalog Role: rw-role (in raw-prod)
          └── Privilege: CATALOG_MANAGE_CONTENT [catalog]
          └── Privilege: CATALOG_READ_PROPERTIES [catalog]
      └── Catalog Role: prop-role (in ml-prod)
          └── Privilege: CATALOG_READ_PROPERTIES [catalog]
      └── Catalog Role: prop-role (in platform-prod)
          └── Privilege: CATALOG_READ_PROPERTIES [catalog]
  └── Principal Role: data-steward
      └── Catalog Role: meta-role (in raw-prod)
          └── Privilege: CATALOG_MANAGE_METADATA [catalog]
          └── Privilege: NAMESPACE_LIST [catalog]
      └── Catalog Role: prop-role (in ml-prod)
          └── Privilege: CATALOG_READ_PROPERTIES [catalog]
      └── Catalog Role: prop-role (in platform-prod)
          └── Privilege: CATALOG_READ_PROPERTIES [catalog]

Principal: audit-service
  └── Principal Role: security-auditor
      └── Catalog Role: prop-role (in raw-prod)
          └── Privil